# Experiment: Detectron2 Keypoints Demo

What this notebook teaches:
- Install Detectron2 in Colab and load a keypoint R-CNN model.
- Run person keypoint inference and visualize predictions.
- Map Detectron2 keypoints to the shared canonical schema.
- Export keypoints and write a mini benchmark artifact.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumeyye-agac/human-pose-estimation-experiments.git"
REPO_DIR = Path("/content/human-pose-estimation-experiments")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Using repo root: {repo_root}")


In [ ]:
def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("opencv-python-headless", "matplotlib", "numpy", "pandas")

# Recommended Detectron2 setup for Colab.
# If this fails in your runtime, see fallback notes below.
try:
    pip_install("torch", "torchvision")
    pip_install("git+https://github.com/facebookresearch/detectron2.git")
except Exception as exc:
    print(f"Detectron2 install warning: {exc}")


If Detectron2 import fails, skip inference in this notebook and use the fallback path in `docs/limitations.md`.


In [ ]:
import json
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np

from posebench.benchmark import BenchmarkConfig, benchmark_backend, write_json
from posebench.export import export_frames_to_csv, export_frames_to_json
from posebench.keypoints_schema import map_tool_keypoints_to_canonical

status_payload = {"tool": "detectron2", "status": "not_measured", "notes": "Detectron2 unavailable"}

try:
    from detectron2 import model_zoo
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor
    from detectron2.utils.visualizer import Visualizer

    detectron2_ready = True
except Exception as exc:
    detectron2_ready = False
    status_payload["notes"] = f"Detectron2 import failed: {exc}"

sample_url = "https://upload.wikimedia.org/wikipedia/commons/9/92/Walking_person.jpg"
sample_path = repo_root / "assets" / "sample_input_walking.jpg"
if not sample_path.exists():
    urllib.request.urlretrieve(sample_url, sample_path)

bgr = cv2.imread(str(sample_path))
if bgr is None:
    raise RuntimeError("Could not load sample image")

if detectron2_ready:
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml")
    predictor = DefaultPredictor(cfg)

    outputs = predictor(bgr)
    instances = outputs["instances"].to("cpu")
    keypoints = instances.pred_keypoints.numpy() if instances.has("pred_keypoints") else None

    if keypoints is None or len(keypoints) == 0:
        raise RuntimeError("No keypoints detected by Detectron2")

    first_person = keypoints[0]
    points = [
        {"x": float(kp[0]), "y": float(kp[1]), "confidence": float(kp[2])}
        for kp in first_person
    ]
    canonical = map_tool_keypoints_to_canonical("detectron2", points, min_confidence=0.1)

    frames = [
        {
            "frame_index": 0,
            "timestamp_ms": 0.0,
            "person_id": 0,
            "tool": "detectron2",
            "schema": "coco17",
            "keypoints": canonical,
        }
    ]
    export_frames_to_csv(frames, repo_root / "results" / "detectron2_demo_keypoints.csv")
    export_frames_to_json(frames, repo_root / "results" / "detectron2_demo_keypoints.json")

    vis = Visualizer(bgr[:, :, ::-1], metadata=None, scale=1.0)
    vis_img = vis.draw_instance_predictions(instances).get_image()
    out_path = repo_root / "assets" / "generated" / "detectron2_demo_overlay.jpg"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), vis_img[:, :, ::-1])

    plt.figure(figsize=(7, 4))
    plt.imshow(vis_img)
    plt.axis("off")
    plt.title("Detectron2 keypoints")
    plt.show()

    class D2Backend:
        name = "detectron2"

        def __init__(self, predictor_obj):
            self.predictor_obj = predictor_obj

        def infer(self, frame: np.ndarray):
            return self.predictor_obj(frame)

    bench_result = benchmark_backend(
        backend=D2Backend(predictor),
        frames=[bgr],
        config=BenchmarkConfig(warmup_frames=4, measured_frames=12, repeat=1),
    )
    write_json(bench_result, repo_root / "results" / "benchmark_raw_detectron2_demo.json")
    status_payload = bench_result
else:
    write_json(status_payload, repo_root / "results" / "benchmark_raw_detectron2_demo.json")

status_payload
